# Phase 1 Detector Fine-Tuning in Colab

This notebook is the first practical step of Phase 1.

Use it to:

- mount Google Drive
- open the project repo from Drive
- install Colab dependencies
- sanity-check the YOLO dataset layout
- fine-tune `yolo26s.pt` on the `plate_number` dataset
- copy the best detector checkpoint to a stable location


## Recommended Baseline

Start with one clean baseline before tuning.

- model: `yolo26s.pt`
- epochs: `80`
- image size: `640`
- batch size: `16`
- patience: `20`

After the baseline run, compare one change at a time such as:

- `imgsz=768`
- `epochs=100`
- `model=yolo26n.pt`


In [5]:
from google.colab import drive  # type: ignore[reportMissingImports]

drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
from pathlib import Path

# Update this if your repo lives in a different Drive folder.
REPO_DIR = Path('/content/drive/MyDrive/plate')

DETECTOR_REQUIREMENTS_PATH = REPO_DIR / 'requirements-detector-colab.txt'
FALLBACK_REQUIREMENTS_PATH = REPO_DIR / 'requirements-colab.txt'
REQUIREMENTS_PATH = DETECTOR_REQUIREMENTS_PATH if DETECTOR_REQUIREMENTS_PATH.exists() else FALLBACK_REQUIREMENTS_PATH
MODEL_NAME = 'yolo26s.pt'
EPOCHS = 80
IMGSZ = 640
BATCH = 16
PATIENCE = 20
PROJECT = REPO_DIR / 'outputs' / 'training' / 'usm_lpr'
BASE_DATA_YAML = REPO_DIR / 'configs' / 'detector_data.yaml'
DATA_YAML = PROJECT / 'detector_data_colab.yaml'
RUN_NAME = 'yolo26s_plate_phase1_baseline'

print('REPO_DIR =', REPO_DIR)
print('REQUIREMENTS_PATH =', REQUIREMENTS_PATH)
print('BASE_DATA_YAML =', BASE_DATA_YAML)
print('DATA_YAML =', DATA_YAML)
print('PROJECT =', PROJECT)


REPO_DIR = /content/drive/MyDrive/plate
REQUIREMENTS_PATH = /content/drive/MyDrive/plate/requirements-colab.txt
DATA_YAML = /content/drive/MyDrive/plate/configs/detector_data.yaml
PROJECT = /content/drive/MyDrive/plate/outputs/training/usm_lpr


## Repo Location Note

This notebook expects the repository to already exist in your Google Drive at `REPO_DIR`.

If that path is missing, copy your repo into Drive or clone it there first, then rerun the setup cells.


In [7]:
assert REPO_DIR.exists(), f'Repo folder not found: {REPO_DIR}'
assert REQUIREMENTS_PATH.exists(), (
    'Requirements file not found. Checked: '
    f'{DETECTOR_REQUIREMENTS_PATH} and {FALLBACK_REQUIREMENTS_PATH}'
)
print('Installing from', REQUIREMENTS_PATH)
%cd /content
%pip install --upgrade pip
%pip install -r {REQUIREMENTS_PATH}


Installing from /content/drive/MyDrive/plate/requirements-colab.txt
/content
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 32.1 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 37.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 100.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 50.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 MB 71.6 MB/s  0:00:006m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 137.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 183.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 158.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 115.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 60.7 MB/s  0:00:00

In [8]:
!nvidia-smi


Fri Apr 10 10:52:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
assert REPO_DIR.exists(), f'Repo folder not found: {REPO_DIR}'
assert REQUIREMENTS_PATH.exists(), (
    'Requirements file not found. Checked: '
    f'{DETECTOR_REQUIREMENTS_PATH} and {FALLBACK_REQUIREMENTS_PATH}'
)
assert BASE_DATA_YAML.exists(), f'Base data YAML not found: {BASE_DATA_YAML}'

print('Repo found:', REPO_DIR)
print('Requirements found:', REQUIREMENTS_PATH)
print('Base data YAML found:', BASE_DATA_YAML)


Repo found: /content/drive/MyDrive/plate
Requirements found: /content/drive/MyDrive/plate/requirements-colab.txt
Data YAML found: /content/drive/MyDrive/plate/configs/detector_data.yaml


In [ ]:
import yaml

data_root = REPO_DIR / 'data'
assert data_root.exists(), f'Data root not found: {data_root}'

PROJECT.mkdir(parents=True, exist_ok=True)
config = yaml.safe_load(BASE_DATA_YAML.read_text(encoding='utf-8')) or {}
assert isinstance(config, dict), f'Expected mapping in {BASE_DATA_YAML}, found: {type(config).__name__}'
config['path'] = str(data_root)
config['train'] = 'images/train'
config['val'] = 'images/val'
config['test'] = 'images/test'
config['names'] = {0: 'plate_number'}

DATA_YAML.parent.mkdir(parents=True, exist_ok=True)
DATA_YAML.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')

print('Wrote Colab dataset YAML to:', DATA_YAML)
print(DATA_YAML.read_text(encoding='utf-8'))


In [10]:
from pathlib import Path

data_root = REPO_DIR / 'data'

for split in ['train', 'val', 'test']:
    image_dir = data_root / 'images' / split
    label_dir = data_root / 'labels' / split
    assert image_dir.exists(), f'Missing image dir: {image_dir}'
    assert label_dir.exists(), f'Missing label dir: {label_dir}'
    images = sorted([p for p in image_dir.iterdir() if p.is_file()])
    labels = sorted([p for p in label_dir.iterdir() if p.is_file()])
    print(f'{split}: images={len(images)} labels={len(labels)}')

    image_stems = {p.stem for p in images}
    label_stems = {p.stem for p in labels}
    missing_labels = sorted(image_stems - label_stems)
    orphan_labels = sorted(label_stems - image_stems)

    print('  missing_labels =', len(missing_labels))
    print('  orphan_labels =', len(orphan_labels))
    if missing_labels:
        print('  example missing label:', missing_labels[0])
    if orphan_labels:
        print('  example orphan label:', orphan_labels[0])

    assert not missing_labels, f'{split} is missing label files; example: {missing_labels[0]}'
    assert not orphan_labels, f'{split} has orphan label files; example: {orphan_labels[0]}'


train: images=329 labels=329
  missing_labels = 0
  orphan_labels = 0
val: images=86 labels=86
  missing_labels = 0
  orphan_labels = 0
test: images=85 labels=85
  missing_labels = 0
  orphan_labels = 0


In [11]:
import yaml

data_root = REPO_DIR / 'data'
config = yaml.safe_load(DATA_YAML.read_text(encoding='utf-8')) or {}
assert isinstance(config, dict), f'Expected mapping in {DATA_YAML}, found: {type(config).__name__}'
names = config.get('names')
if isinstance(names, list):
    names = {index: value for index, value in enumerate(names)}
elif isinstance(names, dict):
    names = {int(key): value for key, value in names.items()}
else:
    raise AssertionError(f"Expected 'names' mapping/list in {DATA_YAML}, found: {names!r}")

assert names.get(0) == 'plate_number', f"Expected class 0 to be 'plate_number', found: {names}"
print('Class mapping OK:', names)

for split in ['train', 'val', 'test']:
    label_dir = data_root / 'labels' / split
    empty_labels = 0
    malformed_rows = []
    invalid_class_ids = []

    for label_path in sorted(label_dir.glob('*.txt')):
        raw_text = label_path.read_text(encoding='utf-8').strip()
        if not raw_text:
            empty_labels += 1
            continue

        for line_number, line in enumerate(raw_text.splitlines(), start=1):
            parts = line.split()
            row_id = f'{label_path.name}:{line_number}'

            if len(parts) != 5:
                malformed_rows.append(f'{row_id} has {len(parts)} values')
                continue

            cls_token, *coord_tokens = parts
            try:
                class_value = float(cls_token)
                coords = [float(token) for token in coord_tokens]
            except ValueError:
                malformed_rows.append(f'{row_id} contains non-numeric values')
                continue

            if not class_value.is_integer():
                malformed_rows.append(f'{row_id} uses non-integer class id {cls_token}')
                continue

            class_id = int(class_value)

            if class_id != 0:
                invalid_class_ids.append(f'{row_id} uses class {class_id}')

            if any(value < 0.0 or value > 1.0 for value in coords):
                malformed_rows.append(f'{row_id} has coordinates outside [0, 1]')

        
    print(f'{split}: empty_labels={empty_labels} malformed_rows={len(malformed_rows)} invalid_class_ids={len(invalid_class_ids)}')
    if malformed_rows:
        print('  example malformed row:', malformed_rows[0])
    if invalid_class_ids:
        print('  example invalid class id:', invalid_class_ids[0])

    assert not malformed_rows, f'{split} has malformed YOLO rows; example: {malformed_rows[0]}'
    assert not invalid_class_ids, f'{split} has invalid class ids; example: {invalid_class_ids[0]}'


Class mapping OK: {0: 'plate_number'}
train: empty_labels=8 malformed_rows=0 invalid_class_ids=0
val: empty_labels=0 malformed_rows=0 invalid_class_ids=0
test: empty_labels=1 malformed_rows=0 invalid_class_ids=0


## Note About Empty YOLO Labels

Some empty YOLO label files may be intentional Roboflow null-image negatives.

Do not automatically delete or relabel empty `.txt` files without checking their source context first.


In [14]:
%%writefile /content/drive/MyDrive/plate/configs/detector_data.yaml
path: /content/drive/MyDrive/plate/data
train: images/train
val: images/val
test: images/test

names:
0: plate_number  

Overwriting /content/drive/MyDrive/plate/configs/detector_data.yaml


In [15]:
from ultralytics import YOLO

model = YOLO(MODEL_NAME)
results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project=str(PROJECT),
    name=RUN_NAME,
    patience=PATIENCE,
    seed=42,
)

RUN_DIR = Path(results.save_dir)
print('Training run directory:', RUN_DIR)

results


Ultralytics 8.4.36 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/plate/configs/detector_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo26s_plate_phase1_baseline2, nbs=64, nms=False, opset=None, optimize=False, optimizer

RuntimeError: Dataset '/content/drive/MyDrive/plate/configs/detector_data.yaml' error ❌ object of type 'NoneType' has no len()

In [ ]:
from pathlib import Path

run_dir = Path(results.save_dir) if 'results' in globals() else None
if run_dir is None or not run_dir.exists():
    candidate_dirs = sorted(
        [path for path in PROJECT.glob(f'{RUN_NAME}*') if path.is_dir()],
        key=lambda path: path.stat().st_mtime,
    )
    assert candidate_dirs, f'No run directories found under {PROJECT} matching {RUN_NAME}*'
    run_dir = candidate_dirs[-1]

best_pt = run_dir / 'weights' / 'best.pt'
assert best_pt.exists(), f'best.pt not found at {best_pt}'
test_eval_name = f'{run_dir.name}_test_eval'
best_model = YOLO(str(best_pt))

test_metrics = best_model.val(
    data=str(DATA_YAML),
    split='test',
    project=str(PROJECT),
    name=test_eval_name,
)

test_metrics.results_dict


In [ ]:
from pathlib import Path
import shutil

run_dir = Path(results.save_dir) if 'results' in globals() else None
if run_dir is None or not run_dir.exists():
    candidate_dirs = sorted(
        [path for path in PROJECT.glob(f'{RUN_NAME}*') if path.is_dir()],
        key=lambda path: path.stat().st_mtime,
    )
    assert candidate_dirs, f'No run directories found under {PROJECT} matching {RUN_NAME}*'
    run_dir = candidate_dirs[-1]

best_pt = run_dir / 'weights' / 'best.pt'
last_pt = run_dir / 'weights' / 'last.pt'
target_dir = REPO_DIR / 'models' / 'detector'
target_dir.mkdir(parents=True, exist_ok=True)
target_best = target_dir / 'best.pt'
target_last = target_dir / 'last.pt'

assert best_pt.exists(), f'best.pt not found at {best_pt}'
shutil.copy2(best_pt, target_best)
if last_pt.exists():
    shutil.copy2(last_pt, target_last)

print('Copied best checkpoint to:', target_best)


In [ ]:
from pathlib import Path

run_dir = Path(results.save_dir) if 'results' in globals() else None
if run_dir is None or not run_dir.exists():
    candidate_dirs = sorted(
        [path for path in PROJECT.glob(f'{RUN_NAME}*') if path.is_dir()],
        key=lambda path: path.stat().st_mtime,
    )
    assert candidate_dirs, f'No run directories found under {PROJECT} matching {RUN_NAME}*'
    run_dir = candidate_dirs[-1]

test_eval_dir = PROJECT / f'{run_dir.name}_test_eval'
print('Run directory:', run_dir)
print('Test evaluation directory:', test_eval_dir)
print('Files:')
for path in sorted(run_dir.glob('*')):
    print('-', path)


## After This Notebook

When fine-tuning finishes:

1. review validation metrics, test metrics, and result plots
2. keep the chosen `best.pt` in `models/detector/`
3. record the main detector run metrics for reporting
4. continue with OCR evaluation using the prepared crop truth files
5. continue with end-to-end evaluation

If you want one tuned comparison run next, the best first follow-up is usually:

- keep `yolo26s.pt`
- change `IMGSZ` from `640` to `768`
- keep the other settings the same
